# NB04 v2 — Hyperparameter Tuning HDBSCAN (Range Diperluas)

**Perubahan dari v1:**
- `min_cluster_size` diperluas ke [50, 75, 100, 150, 200, 300, 500] — v1 max 50 terlalu kecil (semua hasil ≥ 150 cluster)
- `cluster_selection_method` ditambah sebagai parameter: `eom` vs `leaf`
- `k_neighbors` dikurangi ke [15, 20, 25] agar runtime tetap wajar

**Strategi:** Build dense matrix **sekali per k** — reuse untuk semua mcs × ms × csm

**Input:** `output_nb01/embeddings.npy`  
**Output:** `output_nb04/tuning_results_v2.csv`, heatmap, lineplot

---
> **Target:** `n_clusters < 150` (dataset < 150 orang), coverage ≥ 70%  
> Dataset: 11.902 foto → rata-rata ~79 foto/orang → butuh `min_cluster_size` jauh lebih besar dari 50


## 0. Instalasi & Import

In [ ]:
import subprocess, sys

def install_faiss():
    for pkg in ["faiss-gpu-cu12", "faiss-gpu-cu11", "faiss-cpu"]:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✅ {pkg}")
            return pkg
    raise RuntimeError("Gagal install faiss")

try:
    import faiss
    print(f"✅ FAISS sudah ada: {faiss.__version__}")
except ImportError:
    install_faiss()

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "hdbscan", "scikit-learn", "matplotlib", "pandas", "scipy", "-q"],
    check=True
)


In [ ]:
import pickle, time, warnings
from pathlib import Path

import faiss, hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")
print(f"faiss {faiss.__version__}  |  hdbscan {hdbscan.__version__}")
try:
    ngpu = faiss.get_num_gpus()
    print(f"GPU: {ngpu}  {'✅' if ngpu > 0 else '⚠️  CPU only'}")
except Exception:
    print("GPU: tidak tersedia")


## 1. Konfigurasi

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
# ─── PATH ─────────────────────────────────────────────────────────────────────
INPUT_EMB_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb01")
OUTPUT_DIR    = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb04")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Parameter Grid v2 ────────────────────────────────────────────────────────
# min_cluster_size diperluas drastis:
# dataset 11.902 foto / <150 orang → rata-rata ~79 foto/orang
# v1 max=50 terlalu kecil → semua n_clusters ≥ 150
K_NEIGHBORS_GRID          = [15, 20, 25]
MIN_CLUSTER_SIZE_GRID     = [50, 75, 100, 150, 200, 300, 500]
MIN_SAMPLES_GRID          = [5, 10, 15]
CLUSTER_SELECTION_METHODS = ["eom", "leaf"]

TARGET_MAX_CLUSTERS = 150
MIN_COVERAGE        = 70.0   # diturunkan dari 80 → lebih realistis untuk mcs besar

K_MAX = max(K_NEIGHBORS_GRID)
total = (len(K_NEIGHBORS_GRID) * len(MIN_CLUSTER_SIZE_GRID)
         * len(MIN_SAMPLES_GRID) * len(CLUSTER_SELECTION_METHODS))

print(f"Grid v2:")
print(f"  k_neighbors               : {K_NEIGHBORS_GRID}")
print(f"  min_cluster_size          : {MIN_CLUSTER_SIZE_GRID}")
print(f"  min_samples               : {MIN_SAMPLES_GRID}")
print(f"  cluster_selection_method  : {CLUSTER_SELECTION_METHODS}")
print(f"  Total kombinasi           : {total}")
print(f"  Target: n_clusters < {TARGET_MAX_CLUSTERS}, coverage >= {MIN_COVERAGE}%")


## 2. Load Data

In [ ]:
embeddings = np.load(INPUT_EMB_DIR / "embeddings.npy").astype(np.float32)
N, D = embeddings.shape

avg_per_person_est = N / TARGET_MAX_CLUSTERS
print(f"Embeddings : {embeddings.shape}")
print(f"Estimasi rata-rata foto/orang  : {avg_per_person_est:.0f}")
print(f"  → min_cluster_size ideal     : {avg_per_person_est*0.4:.0f} – {avg_per_person_est:.0f}")
print(f"Estimasi dense matrix size     : {N*N*8/1024**3:.2f} GB")


## 3. FAISS — One-Time Search

In [ ]:
t0 = time.time()

emb_f32 = embeddings.copy()
faiss.normalize_L2(emb_f32)

try:
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(D))
    print(f"FAISS GPU ({D} dim)")
except Exception:
    index = faiss.IndexFlatIP(D)
    print(f"FAISS CPU ({D} dim)")

index.add(emb_f32)
sim_all, idx_all = index.search(emb_f32, K_MAX + 1)

# Hapus self-match
sim_all  = sim_all[:, 1:]
idx_all  = idx_all[:, 1:]
dist_all = np.clip(1.0 - sim_all, 0.0, 2.0).astype(np.float32)

print(f"Selesai {time.time()-t0:.1f}s  |  dist_all: {dist_all.shape}")


## 4. Grid Search

Loop:  → build dense matrix →  ×  × 


In [ ]:
results = []
done    = 0
t_start = time.time()

print(f"Grid search v2: {total} kombinasi")
print("─" * 80)

for k in K_NEIGHBORS_GRID:
    # Build sparse + dense untuk k ini (sekali pakai untuk semua mcs/ms/csm)
    dist_k = dist_all[:, :k].astype(np.float32)
    idx_k  = idx_all[:, :k]

    rows   = np.repeat(np.arange(N), k)
    cols   = idx_k.ravel()
    data   = dist_k.ravel()
    sparse = csr_matrix((data, (rows, cols)), shape=(N, N))
    sparse = sparse.maximum(sparse.T)

    dense  = np.full((N, N), 2.0, dtype=np.float64)
    np.fill_diagonal(dense, 0.0)
    cx     = sparse.tocoo()
    dense[cx.row, cx.col] = cx.data.astype(np.float64)

    print(f"
k={k}: dense {dense.nbytes/1024**3:.2f} GB siap")

    for mcs in MIN_CLUSTER_SIZE_GRID:
        for ms in MIN_SAMPLES_GRID:
            for csm in CLUSTER_SELECTION_METHODS:
                t0   = time.time()
                done += 1
                try:
                    clusterer = hdbscan.HDBSCAN(
                        min_cluster_size=mcs,
                        min_samples=ms,
                        metric="precomputed",
                        cluster_selection_method=csm,
                        n_jobs=-1,
                    )
                    clusterer.fit(dense)
                    labels      = clusterer.labels_
                    n_clusters  = int(len(set(labels[labels >= 0])))
                    coverage    = float((labels >= 0).sum() / N * 100)
                    noise_pct   = float((labels == -1).sum() / N * 100)
                    dbcv_approx = float(clusterer.relative_validity_)
                except Exception as e:
                    n_clusters, coverage, noise_pct, dbcv_approx = -1, -1.0, -1.0, float("nan")

                t_run = time.time() - t0
                results.append({
                    "k_neighbors"             : k,
                    "min_cluster_size"        : mcs,
                    "min_samples"             : ms,
                    "cluster_selection_method": csm,
                    "n_clusters"              : n_clusters,
                    "coverage_pct"            : round(coverage, 2),
                    "noise_pct"               : round(noise_pct, 2),
                    "dbcv_approx"             : round(dbcv_approx, 4),
                    "runtime_s"               : round(t_run, 1),
                })

                t_el  = time.time() - t_start
                t_est = t_el / done * (total - done)
                flag  = " ✅" if n_clusters < TARGET_MAX_CLUSTERS and coverage >= MIN_COVERAGE else ""
                print(f"  [{done:3}/{total}] k={k} mcs={mcs:3} ms={ms:2} {csm:4} → "
                      f"n={n_clusters:4}, cov={coverage:5.1f}%, "
                      f"DBCV={dbcv_approx:+.3f}  {t_run:.0f}s  ~{t_est/60:.0f}mnt{flag}")

df = pd.DataFrame(results)
t_total = time.time() - t_start
print(f"
✅ Selesai dalam {t_total:.0f}s ({t_total/60:.1f} menit)")


## 5. Hasil & Analisis

In [ ]:
df_valid = df[df["n_clusters"] > 0].copy()
COLS = ["k_neighbors","min_cluster_size","min_samples","cluster_selection_method",
        "n_clusters","coverage_pct","noise_pct","dbcv_approx"]

print(f"Valid   : {len(df_valid)}/{len(df)}")
print(f"n_clusters range  : {df_valid.n_clusters.min()} – {df_valid.n_clusters.max()}")
print(f"coverage range    : {df_valid.coverage_pct.min():.1f}% – {df_valid.coverage_pct.max():.1f}%")
print(f"DBCV range        : {df_valid.dbcv_approx.min():.3f} – {df_valid.dbcv_approx.max():.3f}")
print()

# Top 5 n_clusters terkecil (tanpa filter)
print("=== 10 hasil dengan n_clusters TERKECIL ===")
print(df_valid.sort_values("n_clusters").head(10)[COLS].to_string(index=False))
print()

# Filter target
df_target = df_valid[
    (df_valid["n_clusters"] < TARGET_MAX_CLUSTERS) &
    (df_valid["coverage_pct"] >= MIN_COVERAGE)
].sort_values("dbcv_approx", ascending=False)

print(f"Memenuhi target (n<{TARGET_MAX_CLUSTERS}, cov≥{MIN_COVERAGE}%) : {len(df_target)}")
if len(df_target) > 0:
    print()
    print("=== TOP 20 (filter target, sort DBCV) ===")
    print(df_target[COLS].head(20).to_string(index=False))


In [ ]:
# ── Heatmap n_clusters: eom vs leaf per k ────────────────────────────────────
n_k   = len(K_NEIGHBORS_GRID)
n_csm = len(CLUSTER_SELECTION_METHODS)
fig, axes = plt.subplots(n_csm, n_k, figsize=(5*n_k, 5*n_csm))
fig.suptitle("n_clusters per (min_cluster_size × min_samples)", fontsize=13, fontweight="bold")

# Cari color range global
vmin = df_valid.n_clusters.min()
vmax = df_valid.n_clusters.max()

for ci, csm in enumerate(CLUSTER_SELECTION_METHODS):
    for ki, k in enumerate(K_NEIGHBORS_GRID):
        ax = axes[ci, ki] if n_csm > 1 else axes[ki]
        df_sub = df_valid[(df_valid.k_neighbors == k) &
                          (df_valid.cluster_selection_method == csm)]
        if df_sub.empty:
            continue
        pivot = df_sub.pivot(index="min_cluster_size", columns="min_samples", values="n_clusters")

        im = ax.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto",
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, fontsize=9)
        ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel("min_samples", fontsize=9)
        ax.set_ylabel("min_cluster_size", fontsize=9)
        ax.set_title(f"csm={csm}  k={k}  [bold=target]", fontsize=10, fontweight="bold")

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    fw = "bold" if val < TARGET_MAX_CLUSTERS else "normal"
                    ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                            fontsize=8, fontweight=fw, color="black")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_nclusters_v2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: heatmap_nclusters_v2.png")


In [ ]:
# ── Line plot: n_clusters & coverage vs min_cluster_size ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = plt.cm.tab20(np.linspace(0, 1, n_k * n_csm))
ms_mid = MIN_SAMPLES_GRID[len(MIN_SAMPLES_GRID)//2]

ci_plot = 0
for csm in CLUSTER_SELECTION_METHODS:
    for k in K_NEIGHBORS_GRID:
        df_line = df_valid[(df_valid.k_neighbors == k) &
                           (df_valid.cluster_selection_method == csm) &
                           (df_valid.min_samples == ms_mid)
                          ].sort_values("min_cluster_size")
        lbl = f"k={k} {csm}"
        lw  = 2.5 if csm == "leaf" else 1.5
        ls  = "-" if csm == "leaf" else "--"
        for ax, metric, target, ylabel in zip(
            axes,
            ["n_clusters", "coverage_pct"],
            [TARGET_MAX_CLUSTERS, MIN_COVERAGE],
            ["Jumlah Cluster", "Coverage (%)"]
        ):
            ax.plot(df_line.min_cluster_size, df_line[metric],
                    marker="o", label=lbl, color=colors[ci_plot],
                    linewidth=lw, linestyle=ls)
        ci_plot += 1

for ax, metric, target, ylabel in zip(
    axes,
    ["n_clusters", "coverage_pct"],
    [TARGET_MAX_CLUSTERS, MIN_COVERAGE],
    ["Jumlah Cluster", "Coverage (%)"]
):
    ax.axhline(target, color="red", linestyle=":", linewidth=2, label=f"Target ({target})")
    ax.set_xlabel("min_cluster_size", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"{ylabel} vs min_cluster_size  (ms={ms_mid})", fontsize=11, fontweight="bold")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lineplot_v2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: lineplot_v2.png")


## 6. Parameter Terbaik

In [ ]:
print("=" * 65)
print("  PARAMETER TERBAIK — NB04 v2")
print("=" * 65)

df_best = df_valid[
    (df_valid.n_clusters < TARGET_MAX_CLUSTERS) &
    (df_valid.coverage_pct >= MIN_COVERAGE)
].sort_values("dbcv_approx", ascending=False)

if len(df_best) > 0:
    best = df_best.iloc[0]
    print(f"  k_neighbors              : {int(best.k_neighbors)}")
    print(f"  min_cluster_size         : {int(best.min_cluster_size)}")
    print(f"  min_samples              : {int(best.min_samples)}")
    print(f"  cluster_selection_method : {best.cluster_selection_method}")
    print()
    print(f"  n_clusters  : {int(best.n_clusters)}")
    print(f"  coverage    : {best.coverage_pct:.2f}%")
    print(f"  noise       : {best.noise_pct:.2f}%")
    print(f"  DBCV approx : {best.dbcv_approx:.4f}")
    print("=" * 65)
    print()
    print("Gunakan di NB02/NB05:")
    print(f"  K_NEIGHBORS               = {int(best.k_neighbors)}")
    print(f"  MIN_CLUSTER_SIZE          = {int(best.min_cluster_size)}")
    print(f"  MIN_SAMPLES               = {int(best.min_samples)}")
    print(f"  CLUSTER_SELECTION_METHOD  = "{best.cluster_selection_method}"")
    best_params = dict(best)
else:
    best_params = {}
    print("⚠️  Masih tidak ada kombinasi memenuhi kriteria.")
    print()
    # Tampilkan apapun yang paling mendekati target
    print("→ Semua hasil diurutkan n_clusters terkecil:")
    print(df_valid.sort_values("n_clusters")[COLS].head(10).to_string(index=False))
    print()
    print("→ Pertimbangkan min_cluster_size > 500 atau ganti pipeline.")


## 7. Simpan Hasil

In [ ]:
csv_path = OUTPUT_DIR / "tuning_results_v2.csv"
pkl_path = OUTPUT_DIR / "tuning_results_v2.pkl"

df.to_csv(csv_path, index=False)
with open(pkl_path, "wb") as f:
    pickle.dump({"results_df": df, "best_params": best_params}, f)

print(f"CSV : {csv_path}  ({csv_path.stat().st_size/1024:.1f} KB)")
print(f"PKL : {pkl_path}  ({pkl_path.stat().st_size/1024:.1f} KB)")
print(f"
Total : {len(df)}  |  Valid : {len(df_valid)}  |  Memenuhi target : {len(df_best) if 'df_best' in dir() else '?'}")
